# Paso 2: Cargar un conjunto de datos

Esto descarga el conjunto de datos de reseñas de películas IMDB. Después de la primera descarga, se carga desde la caché en `~/.cache/huggingface/datasets/`. ✅

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

# Paso 3: Transmitir conjuntos de datos grandes  

Algunos conjuntos de datos son demasiado grandes para caber en el disco. La transmisión los carga fila por fila sin descargar todo el conjunto.

In [1]:

from datasets import load_dataset

dataset = load_dataset( "wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

for example in dataset.take(5):
    print(example["title"])

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Anarchism
Albedo
A
Alabama
Achilles


# Paso 4: Formatos de conjuntos de datos

La biblioteca `datasets` utiliza Apache Arrow por debajo. Puedes convertir a otros formatos dependiendo de lo que necesite tu pipeline.

Para trabajos de IA, Parquet es el mejor formato de almacenamiento. Arrow es con lo que trabajas en memoria. CSV y JSON son para intercambio.

In [1]:

from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb", split="train")

dataset.to_csv("imdb_train.csv")
dataset.to_json("imdb_train.json")
dataset.to_parquet("imdb_train.parquet")

Creating CSV from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

33432823

# Paso 5: Divisiones de datos

Todo proyecto de ML necesita tres divisiones:

- Entrenamiento: El modelo aprende de esto (típicamente 80%)

- Validación: Verificas el progreso durante el entrenamiento (típicamente 10%)

- Prueba: Evaluación final después de completar el entrenamiento (típicamente 10%)

Algunos conjuntos de datos vienen pre-divididos. Cuando no es así, divídelos tú mismo.

Establece siempre una semilla para garantizar la reproducibilidad. La misma semilla produce la misma división cada vez

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb", split="train")

split = dataset.train_test_split(test_size=0.2, seed=42)
train_val = split["train"].train_test_split(test_size=0.125, seed=42)

train_ds = train_val["train"]
val_ds = train_val["test"]
test_ds = split["test"]

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train: 17500, Val: 2500, Test: 5000


# Paso 6: Descargar y almacenar en caché modelos

Los modelos son archivos grandes. La biblioteca `huggingface_hub` se encarga de la descarga y el almacenamiento en caché. 

Los modelos se almacenan en caché en `~/.cache/huggingface/hub/`. Una vez descargados, se cargan instantáneamente en ejecuciones posteriores.

In [1]:

from huggingface_hub import hf_hub_download, snapshot_download

model_path = hf_hub_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    filename="config.json"
)
print(f"Cached at: {model_path}")

model_dir = snapshot_download("sentence-transformers/all-MiniLM-L6-v2")
print(f"Full model at: {model_dir}")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Cached at: /home/minnt22/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Full model at: /home/minnt22/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41


# Paso 7: Gestionar archivos grandes

Los pesos del modelo y los conjuntos de datos grandes no deben guardarse en git.

# Paso 8: Patrones de almacenamiento 

**El almacenamiento local** funciona para conjuntos de datos de menos de ~10 GB. La caché de HF se encarga de esto automáticamente.

**El almacenamiento en la nube** es para todo lo que sea más grande o compartido entre máquinas

In [1]:

import os

local_path = os.path.expanduser("~/.cache/huggingface/datasets/")

# s3_path = "s3://my-bucket/datasets/"
# gcs_path = "gs://my-bucket/datasets/"